# Bybit P2P API Explorer
Usar para descubrir Ad IDs, verificar credenciales y testear endpoints.

## IMPORTANTE: Formato de respuesta
La P2P API de Bybit devuelve **snake_case**: `ret_code`, `ret_msg` (no `retCode`/`retMsg` como el resto de la v5 API).

In [ ]:
import hmac, hashlib, time, requests, json, os
from dotenv import load_dotenv

load_dotenv('../backend/.env')

API_KEY    = os.getenv('BYBIT_API_KEY')
API_SECRET = os.getenv('BYBIT_API_SECRET')
BASE_URL   = os.getenv('BYBIT_BASE_URL', 'https://api.bybit.com')
RECV_WINDOW = 5000

def sign(timestamp, body_str):
    payload = f'{timestamp}{API_KEY}{RECV_WINDOW}{body_str}'
    return hmac.new(API_SECRET.encode(), payload.encode(), hashlib.sha256).hexdigest()

def post(path, body):
    ts       = int(time.time() * 1000)
    body_str = json.dumps(body, separators=(',', ':'))  # compact JSON, no spaces
    headers  = {
        'Content-Type':       'application/json',
        'X-BAPI-API-KEY':     API_KEY,
        'X-BAPI-TIMESTAMP':   str(ts),
        'X-BAPI-RECV-WINDOW': str(RECV_WINDOW),
        'X-BAPI-SIGN':        sign(ts, body_str),
    }
    r = requests.post(f'{BASE_URL}{path}', data=body_str, headers=headers)
    print(f'HTTP {r.status_code}')
    return r.json()

print('API Key cargado:', bool(API_KEY))
print('Base URL:', BASE_URL)

In [ ]:
# ── PASO 1: Listar TUS anuncios activos → obtener Ad IDs ─────────────────────
# Endpoint correcto para mis anuncios: /v5/p2p/item/personal/list

data = post('/v5/p2p/item/personal/list', {'status': '1'})
print('ret_code:', data.get('ret_code'))
items = data.get('result', {}).get('list', data.get('result', {}).get('items', []))
print(f'Anuncios activos: {len(items)}')
for ad in items:
    side_label = 'COMPRA (side=0)' if str(ad.get('side')) == '0' else 'VENTA (side=1)'
    print(f"  [{side_label}]")
    print(f"    id          = {ad['id']}")
    print(f"    price       = {ad['price']}")
    print(f"    quantity    = {ad['quantity']}")
    print(f"    lastQuantity= {ad.get('lastQuantity')}")
    print(f"    minAmount   = {ad['minAmount']}")
    print(f"    maxAmount   = {ad['maxAmount']}")
    print(f"    status      = {ad.get('status')} (10=online, 20=offline)")
    print(f"    payments    = {ad.get('payments')}")
    print(f"    paymentPeriod = {ad.get('paymentPeriod')}")
    print()

In [ ]:
# ── PASO 2: Detalle completo de un anuncio (incluye paymentTerms con los IDs reales) ──

AD_ID = 'PEGA_TU_AD_ID_AQUI'

data = post('/v5/p2p/item/info', {'itemId': AD_ID})
result = data.get('result', {})

print('=== CAMPOS CLAVE PARA EL UPDATE ===')
print(f"  id            = {result.get('id')}")
print(f"  priceType     = {result.get('priceType')}  (0=fixed, 1=floating)")
print(f"  premium       = {result.get('premium')}")
print(f"  price         = {result.get('price')}")
print(f"  minAmount     = {result.get('minAmount')}")
print(f"  maxAmount     = {result.get('maxAmount')}")
print(f"  quantity      = {result.get('quantity')}")
print(f"  lastQuantity  = {result.get('lastQuantity')}  ← usar este para el update")
print(f"  paymentPeriod = {result.get('paymentPeriod')}")
print(f"  status        = {result.get('status')}  (10=online)")
print(f"  remark        = {result.get('remark')}")
print()
print('=== PAYMENT TERMS (usar .id para paymentIds) ===')
for pt in result.get('paymentTerms', []):
    print(f"  id={pt['id']} | paymentType={pt.get('paymentType')}")
print()
print('=== TRADING PREFERENCE SET ===')
print(json.dumps(result.get('tradingPreferenceSet', {}), indent=2))

In [ ]:
# ── PASO 3: Test de actualización de precio (DRY RUN — revisa el body, ejecuta con cuidado) ──
# La API requiere EXACTAMENTE estos campos con estos nombres.
# Cualquier campo faltante o mal nombrado da 404 o error de validación.

AD_ID     = 'PEGA_TU_AD_ID_AQUI'
NEW_PRICE = '1550.50'  # ARS, siempre string

# Fetch detalle actual
detail = post('/v5/p2p/item/info', {'itemId': AD_ID}).get('result', {})

# Extraer paymentIds desde paymentTerms
payment_ids = [str(pt['id']) for pt in detail.get('paymentTerms', [])]
if not payment_ids:
    payment_ids = [str(p) for p in detail.get('payments', [])]

# Normalizar tradingPreferenceSet: todos los valores deben ser strings
raw_prefs = detail.get('tradingPreferenceSet', {})
prefs_str = {k: str(v) for k, v in raw_prefs.items()}

update_body = {
    'id':          str(detail['id']),          # 'id', NO 'itemId'
    'priceType':   str(detail.get('priceType', 0)),
    'premium':     str(detail.get('premium', '')),
    'price':       NEW_PRICE,
    'minAmount':   str(detail['minAmount']),
    'maxAmount':   str(detail['maxAmount']),
    'remark':      detail.get('remark', ''),
    'tradingPreferenceSet': prefs_str,
    'paymentIds':  payment_ids,
    'actionType':  'MODIFY' if detail.get('status') == 10 else 'ACTIVE',
    'quantity':    str(detail.get('lastQuantity') or detail.get('quantity')),
    'paymentPeriod': str(detail.get('paymentPeriod', 15)),
}

print('Body que se enviará a /v5/p2p/item/update:')
print(json.dumps(update_body, indent=2, ensure_ascii=False))

# ⚠️  DESCOMENTÁ PARA EJECUTAR DE VERDAD:
# result = post('/v5/p2p/item/update', update_body)
# print('\nRespuesta:')
# print(json.dumps(result, indent=2))

In [ ]:
# ── PASO 4: Ver mercado P2P USDT/ARS ─────────────────────────────────────────
# side '1' = compradores de USDT → referencia para tu precio de VENTA
# side '0' = vendedores de USDT → referencia para tu precio de COMPRA

def get_market(side, n=5):
    body = {'tokenId': 'USDT', 'currencyId': 'ARS', 'side': str(side), 'size': str(n)}
    data = post('/v5/p2p/item/online', body)
    items = data.get('result', {}).get('items', [])
    for i, item in enumerate(items):
        print(f"  [{i+1}] ${item['price']} | qty={item.get('lastQuantity')} | {item['nickName']}")
    return items

print('=== COMPRADORES USDT (side=1) — referencia para tu VENTA ===')
buyers = get_market(1)
print()
print('=== VENDEDORES USDT (side=0) — referencia para tu COMPRA ===')
sellers = get_market(0)